# GhostWorks Intelligence — Demo Notebook

**Territorial transformation detection + AI interpretation**  
LuxVerso Research Initiative · Vinicius Buri  
ORCID: 0009-0000-6006-1516

---

This notebook demonstrates the full GhostWorks Intelligence pipeline:

1. Load GhostWorks session outputs (TTI pipeline)
2. Serialize into structured territorial context (JSON)
3. Feed to Gemma-family model via Google Cloud Vertex AI
4. Generate territorial intelligence report

**Cases:** Aral Sea (Central Asia) · MATOPIBA (Brazil)

---

### Scientific foundation

TTI (Territorial Transformation Index) is defined as:

```
TTI(x, t₁, t₂) = 1 − cosine_similarity(E(x,t₁), E(x,t₂))
```

Where E is a 64-dimensional embedding from **AlphaEarth Foundation Model** (Google DeepMind).  
Scale: 0.0 (no change) → 1.0 (maximum transformation).  
Label-agnostic: detects change regardless of type.

Reference: Buri, V. (2026). *Territorial Transformation Index: Embedding-based territorial change detection — Application to Brazil (2017–2024)*. Zenodo.

## 0. Setup

In [ ]:
# Mount Google Drive (if running on Colab)
from google.colab import drive
drive.mount('/content/drive')

# Set your session directory
SESSION_DIR = "/content/drive/MyDrive/GhostWorks Exploração"
print(f"Session directory: {SESSION_DIR}")

In [ ]:
!pip install -q google-cloud-aiplatform google-genai

## 1. Serialize GhostWorks Session

In [ ]:
# Copy ghostworks_serializer.py to working directory or paste inline
# Full source: https://github.com/viniburilux/TTI_Brazil_2017_2024

import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# [Paste ghostworks_serializer.py content here or import from repo]
# from ghostworks_serializer import serialize_session

print("Serializer loaded.")

In [ ]:
# Serialize Aral Sea session
ctx_aral = serialize_session("aral_sea", session_dir=SESSION_DIR)
print("Aral Sea context (preview):")
print(ctx_aral[:1500])

In [ ]:
# Serialize MATOPIBA session
ctx_matopiba = serialize_session("matopiba", session_dir=SESSION_DIR)
print("MATOPIBA context (preview):")
print(ctx_matopiba[:1500])

## 2. GhostWorks Intelligence Agent — System Prompt

In [ ]:
SYSTEM_PROMPT = """
You are GhostWorks Intelligence — a territorial analysis agent specialized in
detecting, interpreting, and contextualizing land transformation events from
satellite-derived semantic embeddings.

You operate on the Territorial Transformation Index (TTI):
TTI(x, t₁, t₂) = 1 − cos_similarity(E(x,t₁), E(x,t₂))
Where E is a 64-dimensional AlphaEarth Foundation Model embedding (Google DeepMind).
Scale: 0 (no change) → 1 (maximum transformation). Label-agnostic.

National benchmarks (Brazil, N=10,000): median 0.030 | P90 0.087 | P99 0.209

Always respond with these 7 sections:
1. TERRITORIAL STATUS SUMMARY
2. TEMPORAL DYNAMICS
3. ANOMALY ANALYSIS
4. TRANSFORMATION HYPOTHESES (ranked, with confidence levels)
5. SIMILAR REGION ANALYSIS
6. RISK PROJECTION
7. INVESTIGATIVE RECOMMENDATIONS

Be precise. Cite numerical values. Never invent data.
Write as a senior territorial analyst, not a chatbot.
"""

def build_prompt(ctx_json):
    return f"""
<territorial_data>
{ctx_json}
</territorial_data>

Generate a complete territorial intelligence report for this region.
"""

print("System prompt ready.")

## 3. Generate Intelligence Report via Google Cloud Vertex AI

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google import genai

client = genai.Client(
    vertexai=True,
    project="YOUR_PROJECT_ID",  # Replace with your Google Cloud project ID
    location="global",
)

In [ ]:
# Generate report — Aral Sea
print("=" * 60)
print("GHOSTWORKS INTELLIGENCE REPORT — ARAL SEA")
print("=" * 60)

response_aral = client.models.generate_content(
    model="gemini-2.5-flash",  # Gemma-family via Vertex AI
    contents=SYSTEM_PROMPT + "\n\n" + build_prompt(ctx_aral),
)

print(response_aral.text)

In [ ]:
# Generate report — MATOPIBA
print("=" * 60)
print("GHOSTWORKS INTELLIGENCE REPORT — MATOPIBA")
print("=" * 60)

response_matopiba = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=SYSTEM_PROMPT + "\n\n" + build_prompt(ctx_matopiba),
)

print(response_matopiba.text)

---

## Citation

```
Buri, V. (2026). GhostWorks Intelligence: Territorial transformation detection 
powered by foundation models. LuxVerso Research Initiative.
https://github.com/viniburilux/TTI_Brazil_2017_2024
```

**Primary data:** AlphaEarth Foundation Satellite Embedding dataset (Google + Google DeepMind), CC-BY 4.0  
**License:** CC-BY 4.0